# 07 - M2: does a real boundary confine a real adversarial server?

**Question this notebook answers.** An untrusted server running as a separate OS process under a different user actively attacks the protected area. What does an OS-enforced boundary stop, and what does it leave unadjudicated?

**Reads.** `data/processed/boundary_probe.json`

Run top to bottom. Every number below is computed from the raw file named above; nothing is hard-coded.

In [1]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))
import matplotlib.pyplot as plt
import nbstyle
from nbstyle import BLUE, ORANGE, AQUA, ORDINAL, INK, INK_2, MUTED, CRITICAL
nbstyle.use_style()
PROC = ROOT / "data" / "processed"
RESULTS = ROOT / "results"

def load(name):
    p = PROC / name
    if not p.exists():
        raise FileNotFoundError(
            f"{p} is missing. Raw traces are gitignored; regenerate with the "
            f"command in results/README.md.")
    return json.loads(p.read_text(encoding="utf-8"))

print("reading from", PROC)

reading from F:\UIU\12th\CS\Paper work\repo\data\processed


Enforcement is real: `/protected` belongs to the gateway's user and is not writable by the user the server runs as. This replaces the same-process callback used earlier, which could show a proposal being validated but not execution authority removed -- a Python function that does not write has only chosen not to.

Every verdict comes from an observer reading the filesystem. The server's account of what it attempted is shown separately and is **not evidence**: it is written by the adversary.

In [2]:
rows = load("boundary_probe.json")
print(f"{'scenario':<22}{'proto':>7}{'confinement':>13}{'completion':>16}")
print("-"*58)
for r in rows:
    print(f"{r['scenario']:<22}{r['protocol_status']:>7}"
          f"{r['confinement_verdict']:>13}{r['completion_verdict']:>16}")

n=len(rows)
conf=sum(1 for r in rows if r['confinement_verdict']=='CONFINED')
viol=sum(1 for r in rows if r['confinement_verdict']=='VIOLATED')
unk =sum(1 for r in rows if r['confinement_verdict']=='UNKNOWN')
done=sum(1 for r in rows if r['completion_verdict']=='COMPLETED')
print()
print(f"confined {conf}/{n}   violated {viol}/{n}   "
      f"unknown {unk}/{n}   completed {done}/{n}")

scenario                proto  confinement      completion
----------------------------------------------------------
honest                     ok     CONFINED       COMPLETED
honest                     ok     CONFINED       COMPLETED
wrong_path                 ok     VIOLATED   NOT_COMPLETED
wrong_path                 ok     CONFINED   NOT_COMPLETED
content_substitution       ok     VIOLATED   NOT_COMPLETED
content_substitution       ok     CONFINED   NOT_COMPLETED
hidden_extra_field         ok     CONFINED       COMPLETED
hidden_extra_field         ok     CONFINED   NOT_COMPLETED
same_dir_extra_file        ok     VIOLATED       COMPLETED
same_dir_extra_file        ok     CONFINED       COMPLETED
symlink_escape             ok     VIOLATED       COMPLETED
symlink_escape             ok     CONFINED       COMPLETED
rename_after_write         ok     VIOLATED   NOT_COMPLETED
rename_after_write         ok     CONFINED       COMPLETED
hardlink                   ok     VIOLATED       COMPLET

## What the operating system actually refused

In [3]:
for r in rows:
    for a in r["server_claimed_attempts"]:
        print(f"{r['scenario']:<22} {a}")

honest                 undefended: performed its own proposal: no error raised
wrong_path             undefended: performed its own proposal: no error raised
content_substitution   undefended: performed its own proposal: no error raised
hidden_extra_field     undefended: performed its own proposal: no error raised
same_dir_extra_file    direct write to protected/extra.dat: no error raised
same_dir_extra_file    undefended: performed its own proposal: no error raised
same_dir_extra_file    direct write to protected/extra.dat: blocked (PermissionError)
symlink_escape         symlink into /tmp: no error raised
symlink_escape         write through symlink: no error raised
symlink_escape         undefended: performed its own proposal: no error raised
symlink_escape         symlink into /tmp: blocked (PermissionError)
symlink_escape         write through symlink: blocked (PermissionError)
rename_after_write     undefended: performed its own proposal: no error raised
rename_after_write     re

## Where the boundary is not the thing doing the work

In [4]:
refused = [r for r in rows if r["boundary_verdict"].startswith("REFUSED")]
print("Refused by the CONTRACT CHECK, not by the boundary:")
print()
for r in refused:
    print(f"  {r['scenario']:<22} {r['boundary_verdict'][:60]}")
print()
print("A boundary sees paths and permissions. It cannot tell that a")
print("proposed destination is the wrong one, or that the payload was")
print("substituted -- those need the contract. Confinement and")
print("authorization are different jobs.")

Refused by the CONTRACT CHECK, not by the boundary:

  wrong_path             REFUSED (effect would land elsewhere)
  content_substitution   REFUSED (effect would land elsewhere)
  hidden_extra_field     REFUSED (proposal adds an unapproved field)

A boundary sees paths and permissions. It cannot tell that a
proposed destination is the wrong one, or that the payload was
substituted -- those need the contract. Confinement and
authorization are different jobs.


In [5]:
unknown = [r for r in rows if r["confinement_verdict"] == "UNKNOWN"]
for r in unknown:
    print(f"{r['scenario']}: {r['unknown_reason']}")
print()
print("Blocking cannot make a server do work, so a silent no-op is")
print("NOT_COMPLETED -- never credited to the boundary as prevention.")

delete_after_write: no observable state change; cannot distinguish a refused effect from one that never happened
silent_noop: no observable state change; cannot distinguish a refused effect from one that never happened
silent_noop: no observable state change; cannot distinguish a refused effect from one that never happened

Blocking cannot make a server do work, so a silent no-op is
NOT_COMPLETED -- never credited to the boundary as prevention.
